In [ ]:
#Bài tập ở Lớp

# I found this article very helpful:
# https://www.geeksforgeeks.org/minimax-algorithm-in-game-theory-set-1-introduction/

import os, math

def GetWinner(board):
    """
    Returns the winner in the current board if there is one, otherwise it returns None.
    """
    # horizontal
    if board[0] == board[1] and board[1] == board[2]:
        return board[0]
    elif board[3] == board[4] and board[4] == board[5]:
        return board[3]
    elif board[6] == board[7] and board[7] == board[8]:
        return board[6]
    # vertical
    elif board[0] == board[3] and board[3] == board[6]:
        return board[0]
    elif board[1] == board[4] and board[4] == board[7]:
        return board[1]
    elif board[2] == board[5] and board[5] == board[8]:
        return board[2]
    # diagonal
    elif board[0] == board[4] and board[4] == board[8]:
        return board[0]
    elif board[2] == board[4] and board[4] == board[6]:
        return board[2]

def PrintBoard(board):
    """
    Clears the console and prints the current board.
    """
    os.system('cls' if os.name=='nt' else 'clear')
    print(f'''
    {board[0]}|{board[1]}|{board[2]}
    {board[3]}|{board[4]}|{board[5]}
    {board[6]}|{board[7]}|{board[8]}
    ''')

def GetAvailableCells(board):
    """
    Returns a list of indices containing all available cells in a board.
    """
    available = list()
    for cell in board:
        if cell != "X" and cell != "O":
            available.append(cell)
    return available

def minimax(position, depth, alpha, beta, isMaximizing):
    """
    The AI algorithm responsible for choosing the best move. Returns best value of a move.
    """
    # evaluate current board: if maximizing player won -> return 10
    #                         if minimizing player won -> return -10
    #                         if no one is winning (tie) -> return 0

    # NOTE: Even though the following AI plays perfectly, it might
    #       choose to make a move which will result in a slower victory
    #       or a faster loss. Lets take an example and explain it
    #       Assume that there are 2 possible ways for X to win the game from a give board state.
    #       Move A : X can win in 2 move
    #       Move B : X can win in 4 moves
    #       Our evaluation will return a value of +10 for both moves A and B. Even though the move A
    #       is better because it ensures a faster victory, our AI may choose B sometimes. To overcome
    #       this problem we subtract the depth value from the evaluated score. This means that in case
    #       of a victory it will choose a the victory which takes least number of moves and in case of
    #       a loss it will try to prolong the game and play as many moves as possible.
    winner = GetWinner(position)
    if winner != None:
        return 10 - depth if winner == "X" else -10 + depth
    if len(GetAvailableCells(position)) == 0:
        return 0

    if isMaximizing:
        maxEval = -math.inf
        for cell in GetAvailableCells(position):
            position[cell - 1] = "X"
            Eval = minimax(position, depth + 1, alpha, beta, False)
            maxEval = max(maxEval, Eval)
            alpha = max(alpha, Eval)
            position[cell - 1] = cell
            if beta <= alpha:
                break # prune
        return maxEval
    else:
        minEval = +math.inf
        for cell in GetAvailableCells(position):
            position[cell - 1] = "O"
            Eval = minimax(position, depth + 1, alpha, beta, True)
            minEval = min(minEval, Eval)
            beta = min(beta, Eval)
            position[cell - 1] = cell
            if beta <= alpha:
                break # prune
        return minEval

def FindBestMove(currentPosition, AI):
    """
    This will return the best possible move for the player.
    Will Traverse all cells, evaluate minimax function for all empty cells.
    And return the cell with optimal value.
    Parameters:
        currentPosition (list): The current board to find best move for.
        AI (str): The AI Player ("X" or "O").
    Returns:
        int: Index of best move for the current position.
    """
    bestVal = -math.inf if AI == "X" else +math.inf
    bestMove = -1
    for cell in GetAvailableCells(currentPosition):
        currentPosition[cell - 1] = AI
        moveVal = minimax(currentPosition, 0, -math.inf, +math.inf, False if AI == "X" else True)
        currentPosition[cell - 1] = cell
        if (AI == "X" and moveVal > bestVal):
            bestMove = cell
            bestVal = moveVal
        elif (AI == "O" and moveVal < bestVal):
            bestMove = cell
            bestVal = moveVal
    return bestMove

def main():
    player = input("Play as X or O? ").strip().upper()
    AI = "O" if player == "X" else "X"
    currentGame = [*range(1, 10)]
    # X always starts first.
    currentTurn = "X"
    counter = 0
    while True:
        if currentTurn == AI:
            # NOTE: if the AI starts first, it'll always choose index 0 so to save time you could play it.
            cell = FindBestMove(currentGame, AI)
            currentGame[cell - 1] = AI
            currentTurn = player
        elif currentTurn == player:
            PrintBoard(currentGame)
            while True:
                humanInput = int(input("Enter Number: ").strip())
                if humanInput in currentGame:
                    currentGame[humanInput - 1] = player
                    currentTurn = AI
                    break
                else:
                    PrintBoard(currentGame)
                    print("Cell Not Available.")
        if GetWinner(currentGame) != None:
            PrintBoard(currentGame)
            print(f"{GetWinner(currentGame)} WON!!!")
            break
        counter += 1
        if GetWinner(currentGame) == None and counter == 9:
            PrintBoard(currentGame)
            print("Tie.")
            break

if __name__ == "__main__":
    main()


In [3]:
import math, copy

def initial_state(n=3):
    return [[None for _ in range(n)] for _ in range(n)]

def player(board):
    count_o = sum(row.count('O') for row in board)
    count_x = sum(row.count('X') for row in board)
    return 'O' if count_x > count_o else 'X'

def actions(board):
    moves = set()
    n = len(board)
    for r in range(n):
        for c in range(n):
            if board[r][c] is None:
                moves.add((r, c))
    return moves

def result(board, action):
    r, c = action
    new_board = copy.deepcopy(board)
    new_board[r][c] = player(board)
    return new_board

def winner(board):
    n = len(board)
    # Hàng ngang
    for i in range(n):
        if board[i][0] is not None and all(board[i][j] == board[i][0] for j in range(n)):
            return board[i][0]
    # Hàng dọc
    for j in range(n):
        if board[0][j] is not None and all(board[i][j] == board[0][j] for i in range(n)):
            return board[0][j]
    # Chéo chính
    if board[0][0] is not None and all(board[i][i] == board[0][0] for i in range(n)):
        return board[0][0]
    # Chéo phụ
    if board[0][n-1] is not None and all(board[i][n-1-i] == board[0][n-1] for i in range(n)):
        return board[0][n-1]
    return None

def terminal(board):
    return winner(board) is not None or not any(None in row for row in board)

def evaluate(board):
    score = 0
    for row in board:
        score += row.count('X')
        score -= row.count('O')
    return score

def utility(board):
    res = winner(board)
    if res == 'X': return 1000
    if res == 'O': return -1000
    return evaluate(board)

# --- Minimax với Alpha-Beta ---
def max_value(state, depth, max_depth, alpha, beta):
    if terminal(state) or depth == max_depth:
        return utility(state)
    v = -math.inf
    for act in actions(state):
        v = max(v, min_value(result(state, act), depth+1, max_depth, alpha, beta))
        alpha = max(alpha, v)
        if beta <= alpha:  # cắt tỉa
            break
    return v

def min_value(state, depth, max_depth, alpha, beta):
    if terminal(state) or depth == max_depth:
        return utility(state)
    v = math.inf
    for act in actions(state):
        v = min(v, max_value(result(state, act), depth+1, max_depth, alpha, beta))
        beta = min(beta, v)
        if beta <= alpha:  # cắt tỉa
            break
    return v

def minimax(board, max_depth=3):
    if terminal(board):
        return None
    current_player = player(board)
    best_move = None
    if current_player == 'X':
        v = -math.inf
        for act in actions(board):
            cost = min_value(result(board, act), 1, max_depth, -math.inf, math.inf)
            if cost > v:
                v = cost
                best_move = act
    else:
        v = math.inf
        for act in actions(board):
            cost = max_value(result(board, act), 1, max_depth, -math.inf, math.inf)
            if cost < v:
                v = cost
                best_move = act
    return best_move

def print_board(board):
    for row in board:
        print([cell if cell is not None else ' ' for cell in row])
    print("-" * (len(board)*4))

def main():
    n = int(input("Nhập kích thước bàn cờ (ví dụ 3, 5, 10): "))
    board = initial_state(n)
    print("Bạn là X, AI là O.")
    print_board(board)

    while not terminal(board):
        cur_player = player(board)
        if cur_player == 'X':
            try:
                move = input("Nhập tọa độ (hàng cột, ví dụ: 0 0): ").split()
                action = (int(move[0]), int(move[1]))
                if action not in actions(board):
                    print("Nước đi không hợp lệ!")
                    continue
                board = result(board, action)
            except:
                print("Lỗi nhập liệu!")
                continue
        else:
            print("AI đang tính toán...")
            action = minimax(board, max_depth=3)
            board = result(board, action)
        print_board(board)

    res = winner(board)
    if res:
        print(f"Người chiến thắng là: {res}")
    else:
        print("Trận đấu hòa!")

if __name__ == "__main__":
    main()
0

Nhập kích thước bàn cờ (ví dụ 3, 5, 10): 5
Bạn là X, AI là O.
[' ', ' ', ' ', ' ', ' ']
[' ', ' ', ' ', ' ', ' ']
[' ', ' ', ' ', ' ', ' ']
[' ', ' ', ' ', ' ', ' ']
[' ', ' ', ' ', ' ', ' ']
--------------------
Nhập tọa độ (hàng cột, ví dụ: 0 0): 0 0
['X', ' ', ' ', ' ', ' ']
[' ', ' ', ' ', ' ', ' ']
[' ', ' ', ' ', ' ', ' ']
[' ', ' ', ' ', ' ', ' ']
[' ', ' ', ' ', ' ', ' ']
--------------------
AI đang tính toán...
['X', ' ', ' ', ' ', ' ']
[' ', ' ', ' ', ' ', ' ']
[' ', ' ', ' ', ' ', ' ']
[' ', ' ', ' ', ' ', ' ']
['O', ' ', ' ', ' ', ' ']
--------------------
Nhập tọa độ (hàng cột, ví dụ: 0 0): 0 1
['X', 'X', ' ', ' ', ' ']
[' ', ' ', ' ', ' ', ' ']
[' ', ' ', ' ', ' ', ' ']
[' ', ' ', ' ', ' ', ' ']
['O', ' ', ' ', ' ', ' ']
--------------------
AI đang tính toán...
['X', 'X', ' ', ' ', ' ']
[' ', ' ', ' ', ' ', ' ']
[' ', ' ', ' ', ' ', ' ']
[' ', ' ', ' ', ' ', 'O']
['O', ' ', ' ', ' ', ' ']
--------------------
Nhập tọa độ (hàng cột, ví dụ: 0 0): 0 2
['X', 'X', 'X', ' ', 

In [7]:
!pip install python-chess

In [15]:
!pip install python-chess matplotlib

In [25]:
import chess
import math

# Giá trị quân cờ
piece_values = {
    chess.PAWN: 1,
    chess.KNIGHT: 3,
    chess.BISHOP: 3,
    chess.ROOK: 5,
    chess.QUEEN: 9,
    chess.KING: 0  # Vua không tính điểm, vì mất vua là thua
}

def evaluate(board):
    """Đánh giá bàn cờ: tổng điểm quân trắng - tổng điểm quân đen"""
    score = 0
    for piece_type in piece_values:
        score += len(board.pieces(piece_type, chess.WHITE)) * piece_values[piece_type]
        score -= len(board.pieces(piece_type, chess.BLACK)) * piece_values[piece_type]
    return score

def utility(board):
    """Trả về điểm số cuối cùng nếu ván cờ kết thúc"""
    if board.is_checkmate():
        if board.turn == chess.WHITE:
            return -9999  # Đen thắng
        else:
            return 9999   # Trắng thắng
    elif board.is_stalemate() or board.is_insufficient_material():
        return 0
    return evaluate(board)

def max_value(board, depth, max_depth, alpha, beta):
    if board.is_game_over() or depth == max_depth:
        return utility(board)
    v = -math.inf
    for move in board.legal_moves:
        board.push(move)
        v = max(v, min_value(board, depth+1, max_depth, alpha, beta))
        board.pop()
        alpha = max(alpha, v)
        if beta <= alpha:
            break
    return v

def min_value(board, depth, max_depth, alpha, beta):
    if board.is_game_over() or depth == max_depth:
        return utility(board)
    v = math.inf
    for move in board.legal_moves:
        board.push(move)
        v = min(v, max_value(board, depth+1, max_depth, alpha, beta))
        board.pop()
        beta = min(beta, v)
        if beta <= alpha:
            break
    return v

def minimax(board, max_depth=3):
    """Chọn nước đi tốt nhất cho AI"""
    best_move = None
    if board.turn == chess.WHITE:
        v = -math.inf
        for move in board.legal_moves:
            board.push(move)
            cost = min_value(board, 1, max_depth, -math.inf, math.inf)
            board.pop()
            if cost > v:
                v = cost
                best_move = move
    else:
        v = math.inf
        for move in board.legal_moves:
            board.push(move)
            cost = max_value(board, 1, max_depth, -math.inf, math.inf)
            board.pop()
            if cost < v:
                v = cost
                best_move = move
    return best_move

def main():
    board = chess.Board()
    print("Bạn là Trắng (White), AI là Đen (Black).")
    print(board)

    while not board.is_game_over():
        if board.turn == chess.WHITE:
            move_str = input("Nhập nước đi (ví dụ: e2e4): ")
            try:
                move = chess.Move.from_uci(move_str)
                if move in board.legal_moves:
                    board.push(move)
                else:
                    print("Nước đi không hợp lệ!")
                    continue
            except:
                print("Lỗi nhập liệu!")
                continue
        else:
            print("AI đang tính toán...")
            move = minimax(board, max_depth=3)
            board.push(move)
            print(f"AI đi: {move.uci()}")

        print(board)

    if board.is_checkmate():
        if board.turn == chess.WHITE:
            print("AI thắng!")
        else:
            print("Bạn thắng!")
    else:
        print("Trận đấu hòa!")

if __name__ == "__main__":
    main()


Bạn là Trắng (White), AI là Đen (Black).
r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
P P P P P P P P
R N B Q K B N R
Nhập nước đi (ví dụ: e2e4): f2f3
r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . P . .
P P P P P . P P
R N B Q K B N R
AI đang tính toán...
AI đi: g8h6
r n b q k b . r
p p p p p p p p
. . . . . . . n
. . . . . . . .
. . . . . . . .
. . . . . P . .
P P P P P . P P
R N B Q K B N R
Nhập nước đi (ví dụ: e2e4): e2e4
r n b q k b . r
p p p p p p p p
. . . . . . . n
. . . . . . . .
. . . . P . . .
. . . . . P . .
P P P P . . P P
R N B Q K B N R
AI đang tính toán...
AI đi: h8g8
r n b q k b r .
p p p p p p p p
. . . . . . . n
. . . . . . . .
. . . . P . . .
. . . . . P . .
P P P P . . P P
R N B Q K B N R
Nhập nước đi (ví dụ: e2e4): d1h4
Nước đi không hợp lệ!
Nhập nước đi (ví dụ: e2e4):  d1e2
Lỗi nhập liệu!
Nhập nước đi (ví dụ: e2e4): d1e2
r n b q k b r .
p p p p p p p p
. . . . . . . n
. 